# 01 · Setup de BigQuery — Smart Solutions Dietta

**Team Challenge SQL · Parte II** — Alejandro Dietta

Este notebook crea en Google BigQuery el dataset `smart_solutions_dietta` y las 7 tablas del modelo (ver `docs/er_diagram.png` y `docs/normalizacion.md`):

1. Conexión a BigQuery con la service account.
2. Creación del dataset.
3. Definición de los esquemas de las tablas.
4. Creación de las tablas respetando las dependencias entre claves foráneas.
5. Verificación.

## 1. Configuración y conexión

Las variables se leen del fichero `.env` de la raíz del repositorio (ver `.env.example`). Como este notebook está en `parte_2_modelo_bigquery/notebooks/`, `find_dotenv` busca el `.env` subiendo desde la carpeta actual, y la ruta de las credenciales se resuelve respecto a la raíz del repo.

In [3]:
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account

env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
ROOT = Path(env_path).parent

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = ROOT / os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

print(f"Conectado al proyecto: {client.project}")

Conectado al proyecto: tc-sql-adietta


## 2. Crear el dataset

El dataset se crea en la región `EU` (datos en Europa). Con `exists_ok=True`, si ya existe no se modifica.

In [4]:
dataset = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset.location = "EU"
dataset = client.create_dataset(dataset, exists_ok=True)

print(f"Dataset {dataset.dataset_id} listo (región {dataset.location})")

Dataset smart_solutions_dietta listo (región EU)


## 3. Esquemas de las tablas

Tipos de dato de BigQuery: `INT64`, `STRING`, `FLOAT64`, `BOOL`, `DATE` y `TIMESTAMP`.

- `REQUIRED` = campo obligatorio (`NN` en el diagrama ER).
- `NULLABLE` = campo opcional: `phone`, `shipped_date`, `delivered_date` y `comment`.

El diccionario está ordenado según las dependencias entre tablas: primero las que no apuntan a ninguna otra y después las que tienen claves foráneas hacia las anteriores.

| Orden | Tabla | Depende de |
|---|---|---|
| 1 | `categories` | — |
| 2 | `customers` | — |
| 3 | `products` | `categories` |
| 4 | `orders` | `customers` |
| 5 | `order_items` | `orders`, `products` |
| 6 | `payments` | `orders` |
| 7 | `reviews` | `order_items` |

In [5]:
SCHEMAS = {
    "categories": [
        bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("description", "STRING", mode="REQUIRED"),
    ],
    "customers": [
        bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("first_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("last_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("email", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("phone", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("country", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("city", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("acquisition_channel", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("registration_date", "DATE", mode="REQUIRED"),
    ],
    "products": [
        bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("price", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("cost", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("stock", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("is_active", "BOOL", mode="REQUIRED"),
    ],
    "orders": [
        bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_date", "TIMESTAMP", mode="REQUIRED"),
        bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("shipping_address", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("shipping_city", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("shipping_country", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("shipped_date", "DATE", mode="NULLABLE"),
        bigquery.SchemaField("delivered_date", "DATE", mode="NULLABLE"),
    ],
    "order_items": [
        bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("quantity", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("unit_price", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("discount_pct", "FLOAT64", mode="REQUIRED"),
    ],
    "payments": [
        bigquery.SchemaField("payment_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("payment_method", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("amount", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("payment_date", "TIMESTAMP", mode="REQUIRED"),
    ],
    "reviews": [
        bigquery.SchemaField("review_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("rating", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("comment", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("review_date", "DATE", mode="REQUIRED"),
    ],
}

## 4. Crear las tablas

Se recorren los esquemas en el orden definido. Con `exists_ok=True`, las tablas que ya existan no se modifican.

In [6]:
for table_name, schema in SCHEMAS.items():
    table = bigquery.Table(f"{PROJECT_ID}.{DATASET_ID}.{table_name}", schema=schema)
    client.create_table(table, exists_ok=True)
    print(f"Tabla {table_name} creada")

Tabla categories creada
Tabla customers creada
Tabla products creada
Tabla orders creada
Tabla order_items creada
Tabla payments creada
Tabla reviews creada


## 5. Verificación

Listamos las tablas del dataset para comprobar que se han creado las 7.

In [7]:
tables = [table.table_id for table in client.list_tables(DATASET_ID)]
print(f"{len(tables)} tablas en {DATASET_ID}: {tables}")

7 tablas en smart_solutions_dietta: ['categories', 'customers', 'order_items', 'orders', 'payments', 'products', 'reviews']
